# Coupling graphs for the GAT: Pearson, PLV, AEC-c and Granger, one per subject and band

The GAT's graphs in `subject_cache/` (E1, E2) are |Pearson| and MI between electrodes' **power-spectrum vectors**:
they say how alike two electrodes' spectra are. This notebook builds four **time-domain coupling** graphs instead,
for the four twins pre-registered in `docs/PREREG_2026-11-16.md` (part B).

- **Data:** C3's data. Each subject's first 75 join-free 4 s epochs (300 s), average reference, the cache's five bands.
- **One graph per subject and band**, averaged over the 75 epochs. It comes from that subject's own recording only,
  with no labels, so a test subject's graph is what a clinic would compute from the recording.
- **Self-tests first** (Cell 3): each measure is checked on signals whose answer is known.
- **Reads** `data/derivatives/`, **writes** `subject_cache_fc/fc_graphs.npz`. About 2 min (CPU).

In [1]:
# ===========================================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ===========================================================================
import os, time, warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.signal import hilbert

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

WORK_DIR  = Path.cwd()
DATA_ROOT = WORK_DIR / "data"                 # ds004504 with derivatives/
OUT_DIR   = WORK_DIR / "subject_cache_fc"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SFREQ    = 500.0
EPOCH_N  = 2000              # 4 s
N_EPOCHS = 75                # 300 s: the shortest recording has exactly 75 join-free epochs (as C3)
CHANNELS = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2",
            "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]          # subject_cache node order
BANDS    = {"Delta": (0.5, 4.0), "Theta": (4.0, 8.0), "Alpha": (8.0, 13.0),
            "Beta": (13.0, 30.0), "Gamma": (30.0, 45.0)}                   # the cache's five bands
DECIM    = 2                 # 250 Hz for the band graphs
GC_DECIM = 4                 # 125 Hz for Granger (broadband, the derivatives end at 45 Hz)
GC_ORDER = 10                # 80 ms
FLOOR    = 1e-6
GRAPHS   = ["pearson", "plv", "aec", "granger"]
N_JOBS   = max(1, min(8, (os.cpu_count() or 2) - 2))
print(f"data: {DATA_ROOT}\nout : {OUT_DIR}\nworkers: {N_JOBS}")


data: C:\Projects\Major Project\data
out : C:\Projects\Major Project\subject_cache_fc
workers: 8


In [2]:
# ===========================================================================
# CELL 2 — THE FOUR MEASURES
# ===========================================================================
def pearson(ep):
    """ep (E, C, T) real. |zero-lag Pearson r| per epoch, averaged over epochs."""
    x = ep - ep.mean(-1, keepdims=True)
    x = x / np.sqrt((x ** 2).sum(-1, keepdims=True))
    return np.abs(np.einsum("eit,ejt->eij", x, x)).mean(0)


def plv(an):
    """an (E, C, T) analytic signal. |mean over time of exp(i (phi_i - phi_j))| per epoch, averaged."""
    ph = an / np.abs(an)
    return np.abs(np.einsum("eit,ejt->eij", ph, ph.conj()) / an.shape[-1]).mean(0)


def aec_c(an):
    """an (E, C, T). Orthogonalised amplitude-envelope correlation: for seed i, remove from j the part that is in
    phase with i (zero-lag leakage), correlate the envelopes; average i->j and j->i; |r|; mean over epochs."""
    out = []
    for a in an:                                                   # (C, T)
        unit = a.conj() / np.abs(a)                                # (C, T)
        orth = np.abs(np.imag(a[None, :, :] * unit[:, None, :]))  # orth[i, j] = envelope of j orthogonalised to i
        seed = np.abs(a)[:, None, :]                               # envelope of i
        o = orth - orth.mean(-1, keepdims=True)
        s = seed - seed.mean(-1, keepdims=True)
        with np.errstate(invalid="ignore", divide="ignore"):
            r = (o * s).sum(-1) / np.sqrt((o ** 2).sum(-1) * (s ** 2).sum(-1))
        r = np.nan_to_num(r)                                       # i = j: orthogonal part is 0
        out.append(np.abs((r + r.T) / 2))
    return np.mean(out, 0)


def granger(ep, p=GC_ORDER):
    """ep (E, C, T), broadband. Pairwise Granger F(j -> i) = ln(var(x_i | own past) / var(x_i | own + j's past)),
    least squares pooled over epochs. Returns G[i, j] = influence of j on i."""
    ep = ep - ep.mean(-1, keepdims=True)
    E, C, T = ep.shape
    lag = np.stack([ep[:, :, p - k:T - k] for k in range(1, p + 1)], -1)     # (E, C, T-p, p)
    lag = lag.transpose(1, 0, 2, 3).reshape(C, -1, p)                         # (C, N, p)
    tgt = ep[:, :, p:].transpose(1, 0, 2).reshape(C, -1)                     # (C, N)
    resid = lambda X, y: np.mean((y - X @ np.linalg.lstsq(X, y, rcond=None)[0]) ** 2)
    own = np.array([resid(lag[i], tgt[i]) for i in range(C)])
    G = np.zeros((C, C))
    for i in range(C):
        for j in range(C):
            if i != j:
                G[i, j] = np.log(own[i] / resid(np.hstack([lag[i], lag[j]]), tgt[i]))
    return G


def as_graph(m):
    """Self-loop = the node's strongest connection; floor at FLOOR (the GAT adds log E_ij)."""
    m = np.maximum(np.array(m, dtype=np.float64), FLOOR)
    off = m.copy(); np.fill_diagonal(off, 0)
    np.fill_diagonal(m, np.maximum(off.max(1), FLOOR))
    return m


In [3]:
# ===========================================================================
# CELL 3 — SELF-TESTS ON SIGNALS WHOSE ANSWER IS KNOWN
# ===========================================================================
rng = np.random.default_rng(0)
T, fs = 1000, 250.0
t = np.arange(T) / fs
env = 1 + 0.8 * np.sin(2 * np.pi * 0.7 * t)[None]                          # shared slow envelope
car = lambda ph, f=10.0: np.cos(2 * np.pi * f * t + ph)
ep = rng.standard_normal((20, 6, T)) * 0.05
ep[:, 0] += env[0] * car(0)
ep[:, 1] += env[0] * car(0)                                                # 1 = copy of 0: zero-lag leakage
ep[:, 2] += env[0] * car(np.pi / 2)                                        # 2: same envelope, 90 degrees apart
ep[:, 3] += car(rng.uniform(0, 2 * np.pi, (20, 1)), f=11.3)                # 3: drifting phase, flat envelope
an = hilbert(ep, axis=-1)
P, L, A = pearson(ep), plv(an), aec_c(an)
assert P[0, 1] > 0.95 and P[0, 2] < 0.2, "Pearson: a copy correlates, a 90-degree shift does not"
assert L[0, 1] > 0.95 and L[0, 2] > 0.95 and L[0, 3] < 0.5, "PLV: fixed phase lags lock, random phases do not"
assert A[0, 2] > 0.8 and A[0, 1] < A[0, 2] / 2, "AEC-c: a shared envelope survives, a pure copy is removed"

x = rng.standard_normal((10, 3, 600))
x[:, 1, 3:] += 0.9 * x[:, 0, :-3]                                          # 0 drives 1 with a 3-sample delay
Gt = granger(x, p=5)
assert Gt[1, 0] > 0.3 and Gt[0, 1] < 0.02 and Gt[2, 0] < 0.02, "Granger: only 0 -> 1 is directed"
assert np.allclose(np.diag(as_graph(np.array([[1, .2], [.5, 1]]))), [.2, .5]), "self-loop = strongest edge"
print("self-tests passed: Pearson, PLV, AEC-c (leakage removed), Granger (direction), self-loops")


self-tests passed: Pearson, PLV, AEC-c (leakage removed), Granger (direction), self-loops


In [4]:
# ===========================================================================
# CELL 4 — ONE GRAPH PER SUBJECT AND BAND
# ===========================================================================
def set_file(sid):
    path = DATA_ROOT / "derivatives" / sid / "eeg" / f"{sid}_task-eyesclosed_eeg.set"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def clean_epoch_starts(raw):
    """C3's rule: 4 s epochs on a fixed grid from t = 0, dropping any with an ASR join inside."""
    joins = np.array([a["onset"] for a in raw.annotations if a["description"] == "boundary"]) * SFREQ
    starts = np.arange(0, raw.n_times - EPOCH_N + 1, EPOCH_N)
    inside = (joins[None, :] > starts[:, None]) & (joins[None, :] < starts[:, None] + EPOCH_N)
    return starts[~inside.any(axis=1)]


def subject_graphs(sid):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        raw = mne.io.read_raw_eeglab(set_file(sid), preload=False, verbose="ERROR")
    starts = clean_epoch_starts(raw)[:N_EPOCHS]
    assert len(starts) == N_EPOCHS, sid
    x = raw.get_data(picks=CHANNELS, start=0, stop=int(starts[-1]) + EPOCH_N) * 1e6
    x = x - x.mean(0, keepdims=True)                                       # common average reference
    out = {g: [] for g in GRAPHS}
    for lo, hi in BANDS.values():
        xb = mne.filter.filter_data(x, SFREQ, lo, hi, verbose="ERROR")    # zero-phase FIR, continuous
        an = hilbert(xb, axis=-1)                                          # analytic signal, continuous
        ep = np.stack([xb[:, s:s + EPOCH_N:DECIM] for s in starts])
        ea = np.stack([an[:, s:s + EPOCH_N:DECIM] for s in starts])
        out["pearson"].append(as_graph(pearson(ep)))
        out["plv"].append(as_graph(plv(ea)))
        out["aec"].append(as_graph(aec_c(ea)))
    xg = mne.filter.filter_data(x, SFREQ, None, 45.0, verbose="ERROR")    # anti-alias before 125 Hz
    g = as_graph(granger(np.stack([xg[:, s:s + EPOCH_N:GC_DECIM] for s in starts])))
    out["granger"] = [g] * len(BANDS)                                      # one broadband graph for every band
    return {k: np.stack(v) for k, v in out.items()}                        # each (5, 19, 19)


subjects = [f"sub-{i:03d}" for i in range(1, 89)]
t0 = time.time()
res = Parallel(n_jobs=N_JOBS)(delayed(subject_graphs)(s) for s in subjects)
FC = {g: np.stack([r[g] for r in res]).astype(np.float32) for g in GRAPHS}   # (88, 5, 19, 19)
for g, a in FC.items():
    assert a.shape == (88, len(BANDS), 19, 19) and np.isfinite(a).all() and (a > 0).all(), g
np.savez_compressed(OUT_DIR / "fc_graphs.npz", subjects=np.array(subjects), bands=np.array(list(BANDS)),
                    channels=np.array(CHANNELS), **FC)
print(f"{len(subjects)} subjects in {time.time() - t0:.0f} s -> {OUT_DIR / 'fc_graphs.npz'}")


88 subjects in 164 s -> C:\Projects\Major Project\subject_cache_fc\fc_graphs.npz


In [5]:
# ===========================================================================
# CELL 5 — WHAT THE GRAPHS LOOK LIKE (no labels used)
# ===========================================================================
iu = np.triu_indices(19, 1)
rows = []
for g, a in FC.items():
    for b, band in enumerate(BANDS):
        off = a[:, b][:, iu[0], iu[1]] if g != "granger" else a[:, b][:, ~np.eye(19, dtype=bool)]
        rows.append(dict(graph=g, band=band, median=np.median(off), p10=np.percentile(off, 10),
                         p90=np.percentile(off, 90)))
        if g == "granger":
            break
print("off-diagonal edge weights over all subjects\n")
print(pd.DataFrame(rows).round(4).to_string(index=False))
_flat = {g: FC[g][:, 2][:, iu[0], iu[1]].ravel() for g in GRAPHS}
print("\nSpearman correlation between graph types, alpha band edges:")
print(pd.DataFrame(_flat).corr("spearman").round(2).to_string())


off-diagonal edge weights over all subjects

  graph  band  median    p10    p90
pearson Delta  0.3737 0.2280 0.6341
pearson Theta  0.4079 0.1693 0.7190
pearson Alpha  0.4474 0.1719 0.7635
pearson  Beta  0.3275 0.1144 0.6319
pearson Gamma  0.2964 0.1143 0.6027
    plv Delta  0.3824 0.2759 0.5861
    plv Theta  0.3723 0.2018 0.6376
    plv Alpha  0.4157 0.2130 0.6971
    plv  Beta  0.2779 0.1250 0.5385
    plv Gamma  0.2520 0.1217 0.5085
    aec Delta  0.1624 0.1370 0.2097
    aec Theta  0.1372 0.1146 0.1779
    aec Alpha  0.1397 0.1098 0.1948
    aec  Beta  0.0765 0.0621 0.1013
    aec Gamma  0.0756 0.0628 0.1024
granger Delta  0.0167 0.0049 0.0567

Spearman correlation between graph types, alpha band edges:
         pearson   plv   aec  granger
pearson     1.00  0.96  0.12     0.48
plv         0.96  1.00  0.24     0.51
aec         0.12  0.24  1.00     0.16
granger     0.48  0.51  0.16     1.00
